In [ ]:
!pip install -q faster-whisper fastapi uvicorn python-multipart pyngrok torch torchaudio

In [ ]:
# DeepFilterNet3 (speech enhancement) agora usa o binário pré-compilado
# `deep-filter` (musl estático, v0.5.6) em vez do pacote pip `deepfilternet`.
#
# Motivo: o pacote importa `deepfilterlib` (frontend STFT em Rust), que só
# publica wheels até Python 3.11. O Colab usa Python 3.12+ — o pip tentaria
# compilar o Rust do source (precisa toolchain cargo, que não existe no
# Colab) e falharia com "metadata-generation-failed".
#
# O binário roda em qualquer Python, faz o resample 48k<->SR original sozinho
# e reduz ruído forte sem os artefatos do resemble-enhance. Os assets são
# baixados sob demanda na primeira chamada a /enhance (binário ~36MB +
# modelo ONNX ~8MB) e cacheados em ~/.aivideocut/deepfilter.
#
# Não é preciso instalar nada aqui. Se algo falhar, /transcribe continua
# funcionando — só o /enhance ficaria indisponível.

In [ ]:
%%writefile app.py
import os
import shutil
import subprocess
import tempfile
import time
import urllib.request
from contextlib import suppress
from typing import Optional

import numpy as np
import soundfile as sf
import torch
from fastapi import BackgroundTasks, Depends, FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from faster_whisper import WhisperModel

TRANSCRIBE_MODEL_NAME = "large-v3-turbo"
ENHANCE_MODEL_NAME = "DeepFilterNet3"
MAX_UPLOAD_MB = 500

# O /enhance usa o binário pré-compilado `deep-filter` (musl estático) em vez do
# pacote pip deepfilternet: esse pacote importa deepfilterlib (frontend STFT em
# Rust), que só publica wheels até Python 3.11. No Colab (Python 3.12+) o pip
# tentaria compilar o Rust do source e falharia (metadata-generation-failed).
# O binário roda em qualquer Python, faz resample 48k<->SR original sozinho e
# é baixado (binário ~36MB + modelo ~8MB) uma única vez e cacheado em disco.
ENHANCE_BIN_URL = (
    "https://github.com/Rikorose/DeepFilterNet/releases/download/"
    "v0.5.6/deep-filter-0.5.6-x86_64-unknown-linux-musl"
)
ENHANCE_MODEL_URL = (
    "https://raw.githubusercontent.com/Rikorose/DeepFilterNet/main/"
    "models/DeepFilterNet3_onnx.tar.gz"
)
ENHANCE_ASSET_DIR = os.path.join(os.path.expanduser("~"), ".aivideocut", "deepfilter")
# config.ini do DeepFilterNet3: sr=48k, fft_size=960, hop_size=480, lookahead=2
# delay algorítmico = (fft - hop) + lookahead*hop = 480 + 960 = 1440 amostras @48k.
ENHANCE_DELAY_48K = 1440

API_TOKEN = os.environ["API_TOKEN"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"


def _remove_file(path: str) -> None:
    with suppress(OSError):
        os.remove(path)


def _ensure_enhance_assets() -> tuple:
    bin_path = os.path.join(ENHANCE_ASSET_DIR, "deep-filter")
    model_path = os.path.join(ENHANCE_ASSET_DIR, "DeepFilterNet3_onnx.tar.gz")
    os.makedirs(ENHANCE_ASSET_DIR, exist_ok=True)
    if not os.path.isfile(bin_path):
        print(f"Baixando binário deep-filter ({ENHANCE_MODEL_NAME})...", flush=True)
        urllib.request.urlretrieve(ENHANCE_BIN_URL, bin_path)
        os.chmod(bin_path, 0o755)
    if not os.path.isfile(model_path):
        print(f"Baixando modelo {ENHANCE_MODEL_NAME}...", flush=True)
        urllib.request.urlretrieve(ENHANCE_MODEL_URL, model_path)
    return bin_path, model_path

app = FastAPI(
    title="Speech API",
    version="1.0.0"
)

# ==========================
# CORS
# ==========================

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ==========================
# AUTH
# ==========================

bearer_scheme = HTTPBearer()


def require_api_token(
    credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme),
) -> None:
    if credentials.credentials != API_TOKEN:
        raise HTTPException(status_code=401, detail="Invalid or missing API token")


print(f"Carregando modelo de transcrição {TRANSCRIBE_MODEL_NAME} ({DEVICE}/{COMPUTE_TYPE})...")

transcribe_model = WhisperModel(
    TRANSCRIBE_MODEL_NAME,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)

print("Modelo de transcrição carregado!")

# O enhance é carregado sob demanda dentro do endpoint /enhance (não aqui no
# startup): assim, /transcribe e /enhance ficam independentes entre si — se o
# download do binário/modelo DeepFilterNet3 falhar, isso não derruba a API de
# transcrição, e vice-versa.

# ==========================
# ROOT
# ==========================

@app.get("/")
def root():
    return {
        "name": "Speech API",
        "status": "online",
        "version": "1.0.0"
    }

# ==========================
# HEALTH
# ==========================

@app.get("/api/health")
def health():
    return {
        "status": "ok",
        "device": DEVICE,
        "compute_type": COMPUTE_TYPE,
        "version": "1.0.0",
        "endpoints": {
            "transcribe": {"model": TRANSCRIBE_MODEL_NAME, "loaded": True},
            "enhance": {"model": ENHANCE_MODEL_NAME, "loaded": "lazy"}
        }
    }

# ==========================
# TRANSCRIBE
# ==========================

@app.post("/transcribe", dependencies=[Depends(require_api_token)])
def transcribe(
    file: UploadFile = File(...),
    language: str = Form("pt"),
    vad: bool = Form(True),
    word_timestamps: bool = Form(True)
):
    start = time.time()

    suffix = os.path.splitext(file.filename)[1]
    content = file.file.read()

    max_bytes = MAX_UPLOAD_MB * 1024 * 1024
    if len(content) > max_bytes:
        raise HTTPException(
            status_code=413,
            detail=f"File too large (max {MAX_UPLOAD_MB}MB)",
        )

    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(content)
        path = tmp.name

    try:
        segments, info = transcribe_model.transcribe(
            path,
            language=None if language in ("", "auto") else language,
            vad_filter=vad,
            word_timestamps=word_timestamps
        )

        result_segments = []
        full_text = []

        words_count = 0

        duration = 0

        for seg in segments:

            duration = seg.end

            full_text.append(seg.text)

            words = []

            if seg.words:

                for w in seg.words:

                    words_count += 1

                    words.append({
                        "word": w.word,
                        "start": w.start,
                        "end": w.end
                    })

            result_segments.append({

                "start": seg.start,
                "end": seg.end,
                "text": seg.text,
                "words": words

            })
    except Exception as exc:
        raise HTTPException(
            status_code=422, detail=f"Transcription failed: {exc}"
        ) from exc
    finally:
        with suppress(OSError):
            os.remove(path)

    return {

        "success": True,

        "language": info.language,

        "duration": duration,

        "processing_time": round(time.time() - start, 2),

        "model": TRANSCRIBE_MODEL_NAME,

        "segments": result_segments,

        "segments_count": len(result_segments),

        "words_count": words_count,

        "text": "".join(full_text)

    }

# ==========================
# ENHANCE
# ==========================

@app.post("/enhance", dependencies=[Depends(require_api_token)])
def enhance(
    file: UploadFile = File(...),
    denoise_only: bool = Form(False),
    atten_lim_db: Optional[float] = Form(None),
):
    start = time.time()

    suffix = os.path.splitext(file.filename)[1]
    content = file.file.read()

    max_bytes = MAX_UPLOAD_MB * 1024 * 1024
    if len(content) > max_bytes:
        raise HTTPException(
            status_code=413,
            detail=f"File too large (max {MAX_UPLOAD_MB}MB)",
        )

    bin_path, model_path = _ensure_enhance_assets()

    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(content)
        input_path = tmp.name

    workdir = tempfile.mkdtemp(prefix="df_enhance_")
    in_dir = os.path.join(workdir, "in")
    out_dir = os.path.join(workdir, "out")
    os.makedirs(in_dir, exist_ok=True)
    os.makedirs(out_dir, exist_ok=True)
    output_path = f"{input_path}_enhanced.flac"

    try:
        # Decodifica qualquer formato pra wav float32 mono no SR original. O
        # binário deep-filter resampleia pra 48kHz (SR do modelo) e de volta.
        try:
            data, orig_sr = sf.read(input_path, dtype="float32")
        except Exception:
            import torchaudio
            audio, orig_sr = torchaudio.load(input_path)
            data = audio.mean(dim=0).numpy()
        if data.ndim > 1:
            data = data.mean(axis=1)

        # Padding do delay algorítmico do modelo para a saída manter o mesmo
        # tamanho da entrada quando usamos -D (compensate delay).
        pad = int(round(ENHANCE_DELAY_48K * orig_sr / 48000))
        padded = np.concatenate([data, np.zeros(pad, dtype=np.float32)])

        in_wav = os.path.join(in_dir, "input.wav")
        sf.write(in_wav, padded, orig_sr, format="WAV")

        # denoise_only → default do DeepFilterNet (limite de atenuação ~10dB).
        if denoise_only:
            atten_lim_db = 10.0 if atten_lim_db is None else atten_lim_db
        atten = atten_lim_db if atten_lim_db is not None else 100.0

        cmd = [
            bin_path,
            "-m", model_path,
            "-D",
            "-a", str(atten),
            "-o", out_dir,
            in_wav,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
        if result.returncode != 0:
            raise RuntimeError(f"deep-filter falhou: {result.stderr[-2000:]}")

        enhanced, srn = sf.read(os.path.join(out_dir, "input.wav"), dtype="float32")
        sf.write(output_path, enhanced, srn, format="FLAC")
    except Exception as exc:
        import traceback as _tb
        print("\n===== /enhance TRACEBACK =====", flush=True)
        _tb.print_exc()
        with suppress(OSError):
            os.remove(output_path)
        raise HTTPException(
            status_code=422,
            detail={
                "error": "Enhance failed",
                "message": str(exc),
                "type": type(exc).__name__,
                "traceback": _tb.format_exc(),
            },
        ) from exc
    finally:
        with suppress(OSError):
            os.remove(input_path)
        shutil.rmtree(workdir, ignore_errors=True)

    cleanup = BackgroundTasks()
    cleanup.add_task(_remove_file, output_path)

    return FileResponse(
        output_path,
        media_type="audio/flac",
        filename="enhanced.flac",
        background=cleanup,
        headers={
            "X-Model": ENHANCE_MODEL_NAME,
            "X-Processing-Time": str(round(time.time() - start, 2)),
        },
    )


In [ ]:
import os

from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

# Crie um secret "API_TOKEN" no Colab (ícone de chave na barra lateral) com
# um valor aleatório seu — ele protege os endpoints /transcribe e /enhance
# de uso por qualquer pessoa que descubra a URL pública do ngrok.
os.environ['API_TOKEN'] = userdata.get('API_TOKEN')

public_url = ngrok.connect(8000)

print(public_url)
print('Header em ambos os endpoints: Authorization: Bearer <seu API_TOKEN>')
print('POST /transcribe -> transcrição (faster-whisper)')
print('POST /enhance    -> speech enhancement (DeepFilterNet3), independente do /transcribe')

In [ ]:
import subprocess
import time

log_file = open('uvicorn.log', 'w')
uvicorn_process = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

time.sleep(3)
print(f'Uvicorn iniciado (PID {uvicorn_process.pid}). Logs em uvicorn.log')
print(f'Ainda rodando: {uvicorn_process.poll() is None}')
# Pra ver os logs a qualquer momento: !tail -n 50 uvicorn.log
# Pra derrubar o servidor: uvicorn_process.terminate()